## AuxTel Tracking tests

Craig Lage 22-Mar-21

Mike Warner requested tracking tests in the day with closed dome.\
Near the SCP with very slow tracking speeds

In [1]:
import sys, time, os, asyncio

from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from lsst.ts import salobj
from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS
from lsst.ts.observatory.control.utils import RotType
from astropy.time import Time, TimeDelta
from astropy.coordinates import AltAz, ICRS, EarthLocation, Angle, FK5
import astropy.units as u

/opt/lsst/software/stack/conda/miniconda3-py38_4.9.2/envs/lsst-scipipe-0.4.1/lib/python3.8/site-packages/jose/backends/cryptography_backend.py:23: CryptographyDeprecationWarning: int_from_bytes is deprecated, use int.from_bytes instead
  from cryptography.utils import int_from_bytes, int_to_bytes


In [2]:
# Set Cerro Pachon location
location = EarthLocation.from_geodetic(lon=-70.747698*u.deg,
                                       lat=-30.244728*u.deg,
                                       height=2663.0*u.m)

In [3]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [4]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG
# Make matplotlib less chatty
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [5]:
#get classes and start them
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Read historical data in 0.01 sec
Read 100 history items for RemoteEvent(ATHeaderService, 0, heartbeat)
Read 42 history items for RemoteEvent(ATHeaderService, 0, largeFileObjectAvailable)
Read 100 history items for RemoteEvent(ATHeaderService, 0, logMessage)
Read 6 history items for RemoteEvent(ATHeaderService, 0, summaryState)
Read historical data in 0.04 sec
Read 100 history items for RemoteEvent(ATArchiver, 0, heartbeat)
Read 36 history items for RemoteEvent(ATArchiver, 0, imageInOODS)
Read 36 history items for RemoteEvent(ATArchiver, 0, imageRetrievalForArchiving)
Read 2 history items for RemoteEvent(ATArchiver, 0, summa

[[None, None, None, None, None, None, None], [None, None, None, None]]

mountStatus DDS read queue is full (100 elements); data may be lost
guidingAndOffsets DDS read queue is full (100 elements); data may be lost
currentTargetStatus DDS read queue is full (100 elements); data may be lost


In [26]:
# enable components if required
# Failed
await atcs.enable()
await latiss.enable()

Enabling all components
Gathering settings.
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Couldn't get settingVersions event. Using empty settings.
Complete settings for atmcs.
Complete settings for atptg.
Complete settings for ataos.
Complete settings for atpneumatics.
Complete settings for athexapod.
Complete settings for atdome.
Complete settings for atdometrajectory.
Settings versions: {'atmcs': '', 'atptg': '', 'ataos': '', 'atpneumatics': '', 'athexapod': '', 'atdome': '', 'atdometrajectory': ''}
[atmcs]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[atptg]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[ataos]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[atpneumatics]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
Unable to trans

RuntimeError: Failed to transition ['athexapod', 'atdome'] to <State.ENABLED: 2>.

In [ ]:
# Everything now enabled 11:45 AM

In [27]:
await salobj.set_summary_state(atcs.rem.athexapod, salobj.State.ENABLED, settingsToApply='current')

[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]

In [29]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.ENABLED, settingsToApply='current')

[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]

In [25]:
await salobj.set_summary_state(latiss.rem.atspectrograph, salobj.State.ENABLED, settingsToApply='current')

[<State.DISABLED: 1>, <State.ENABLED: 2>]

In [15]:
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [30]:
await atcs.shutdown()

Disabling ATAOS corrections
Disabling ATAOS corrections.
Closing M1 cover vent gates.
Cover state <MirrorCoverState.CLOSED: 6>
M1 cover already closed.
M1 vent state <VentsPosition.CLOSED: 1>
M1 vents already closed.
Close dome.
ATDome Shutter Door is already closed. Ignoring.
Slew dome to Park position.
Sending ATDomeTrajectory to DISABED state. Component will be left in DISABLEDstate or else it may send the ATDome back to alignment with the telescope.
process as completed...
atdometrajectory: <State.DISABLED: 1>
Got True
Waiting for telescope to settle.
[Dome] delta Az = +000.240 deg
ATDome in position.
Axes in position.
Disable ATDomeTrajectory
Slew telescope to Park position.
Sending command
Stop tracking.
Unknown tracking state: 10
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
Got False
Telescope not in position
[Telescope] delta Alt = 

In [ ]:
# turn on ATAOS corrections just to make sure the mirror is under air
tmp = await atcs.rem.ataos.cmd_enableCorrection.set_start(m1=True, hexapod=True, atspectrograph=False)

In [ ]:
atcs.slew_icrs?

In [ ]:
# Slew to starting position and take an image to check headers and tracking
await atcs.slew_icrs(ra=str(ra_dec.ra), dec=str(ra_dec.dec), rot=0.0, rot_type=RotType.PhysicalSky,
                      slew_timeout=240., stop_before_slew=False, wait_settle=True)

await asyncio.sleep(2)
# take a 60s dark
await latiss.take_darks(60.0, 1)


In [ ]:
# Now loop from dec of 70 to almost 90
# RA going from 2h to 6h
for dec in [70.0, 75.0, 80.0, 82.0, 84.0, 86.0, 88.0, 89.0]:
    for ra in ['02:00:00.0', '02:15:00.0', '02:30:00.0', '02:45:00.0', '03:00:00.0', '03:15:00.0',\
                '03:30:00.0', '03:45:00.0', '04:00:00.0', '04:15:00.0', '04:30:00.0', '04:45:00.0',\
                '05:00:00.0', '05:15:00.0', '05:30:00.0', '05:45:00.0', '06:00:00.0']
        await atcs.slew_icrs(ra=ra, dec=dec, rot=-110.0, rot_type=RotType.PhysicalSky,
                              slew_timeout=240., stop_before_slew=False, wait_settle=True)

        await asyncio.sleep(2)
        # take a 60s dark
        await latiss.take_darks(60.0, 1)

In [ ]:
# For shutdown of system
await atcs.stop_tracking()

In [ ]:
# turn off corrections
tmp = await atcs.rem.ataos.cmd_disableCorrection.set_start(m1=True, hexapod=True, atspectrograph=True)

In [ ]:
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [ ]:
# Putting everything back in standby.
# Failed as it usually does
await atcs.shutdown()

In [ ]:
await atcs.rem.atdome.cmd_start.set_start(settingsToApply="test", timeout=30)

In [ ]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.STANDBY, settingsToApply="test")

In [31]:
await salobj.set_summary_state(latiss.rem.atspectrograph, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atcamera, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atheaderservice, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atarchiver, salobj.State.STANDBY)

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [ ]:
# Everything back in standby